### Imports

In [ ]:
from dask.distributed import Client
client = Client()  # your clienti0;oh8uli n
for w in client.scheduler_info()['workers'].values():
    print(f"Worker: {w['name']}, Threads: {w['nthreads']}, Memory limit: {w['memory_limit']}")

In [ ]:
import pickle
import numpy as np
import pandas as pd
import scipy.stats as ss
import statsmodels.api as sm
import dask
import matplotlib.pyplot as plt
import xarray as xr             
import cartopy.crs as ccrs      
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import os
import cartopy.mpl.ticker as cticker
import netCDF4
import glob
import re

from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
from matplotlib.patches import Patch
import matplotlib.colors as colors
import matplotlib
from IPython.display import HTML, display
from pathlib import Path

matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
print("ffmpeg path set to:", imageio_ffmpeg.get_ffmpeg_exe())
from pandas.plotting import register_matplotlib_converters #function used in plotting when converting NetCDF data to dates 
register_matplotlib_converters() #needed to facilitate conversion of dates in netCDF files
%matplotlib inline

### Load in Mean Time Series and Component Datasets

In [ ]:
# Enter your directory
your_dir = "path/to/data"

import sys
sys.path.append(your_dir + '/.local/lib/python3.10/site-packages')

In [ ]:
#import mean time series
des_glob_meanz = np.empty((4,10), dtype='object')

input_dir = your_dir + "/data/Means/60N-60S" #change to /60N-60S for the zonal mean

for j in range(0,1):
    for i in range(10):
        filename = f"glob_mean_var{j}_ds{i}.nc"
        filepath = os.path.join(input_dir, filename)
        if os.path.exists(filepath):
            glob_mean = xr.open_dataarray(filepath)[:].sortby('time').sel(time=slice("2002-09-01", "2023-08-31"))
            if i == 9:
                glob_mean = glob_mean.groupby("time").mean()
            if i == 8:
                glob_mean = glob_mean.groupby("time").mean()
            month_avg = glob_mean.groupby('time.month').mean(dim='time')
            des_glob_meanz[j][i] = glob_mean.groupby('time.month') - month_avg
            first_time = des_glob_meanz[j][i]['time'][0].values
            final_time = des_glob_meanz[j][i]['time'][-1].values
            print("Final time for file", filename, ":", first_time, final_time)
        else:
            des_glob_meanz[j][i] = None
    print("finished variable", j)

#switch BEST and JRA3Q for consistency
for j in range(0,1):
    temporary = des_glob_meanz[j][5]
    des_glob_meanz[j][5] = des_glob_meanz[j][6]
    des_glob_meanz[j][6] = temporary

#sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'IR-only AIRSv6','Skin AIRSv7']

sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']

In [ ]:
# replace GISTEMP with the GISTEMP global average for consistency. ONLY USE FOR GLOBAL MEAN, NOT 60N-60S

df = pd.read_csv(your_dir + '/data/GISTEMP/GLB.Ts+dSST.csv', skiprows=1)
month_cols = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df = df[['Year'] + month_cols]
df = df.replace('***', pd.NA)

for col in month_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
# Convert wide format to long format
long_df = df.melt(id_vars='Year', var_name='Month', value_name='Anomaly')
# Build a datetime index at month start
month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
long_df['MonthNum'] = long_df['Month'].map(month_map)
long_df['Date'] = pd.to_datetime(dict(year=long_df['Year'], month=long_df['MonthNum'], day=1))

ts = long_df.set_index('Date')['Anomaly'].sort_index()

da = xr.DataArray(data=ts.values,dims=['time'],coords={'time': ts.index.values, 'month': ('time', ts.index.month) })
da = da.sel(time=slice('2002-09-01', '2023-08-01'))
da = da.groupby('time.month') - da.groupby('time.month').mean(dim='time')


des_glob_meanz[0][7] = da.copy()

In [ ]:
#Import Datasets Helper Function

def open_sesame(filename):
    path = ryour_dir + '/data/Fitting Components/'
    with open(path+filename+'.txt', "r") as f:
        lines = f.readlines()
    # Skip the first header line
    data_lines = [line for line in lines if line.strip() and line.strip()[0].isdigit()]
    # Parse year + 12 monthly values
    records = []
    for line in data_lines:
        parts = line.strip().split()
        if len(parts) == 13:  # year + 12 months
            year = int(parts[0])
            monthly_vals = [float(val) for val in parts[1:]]
            for month, val in enumerate(monthly_vals, 1):
                records.append((f"{year}-{month:02d}", val))
    
    # Convert to x-array
    df = pd.DataFrame(records, columns=["time", "amo"])
    df["time"] = pd.to_datetime(df["time"])
    df["amo"] = df["amo"].replace(-99.99, np.nan)
    ds = df.set_index("time").to_xarray()
    cut_ds = ds.sel(time=slice("2002-09-01", "2025-08-31"))
    return cut_ds['amo']

In [ ]:
#Import Datasets 

#filenames = ['aao.data', 'amon.us.data', 'ao.data','enso.data','nao.data','pdo.data','qbo.data','iod.data', 'solar.data']
filenames = ['aao.data', 'amon.us.data', 'ao.data','enso.data','nao.data','pdo.data','qbo.data','iod.data', 'solar.data']
#comp_labelz = ['aao', 'amo', 'ao', 'enso', 'nao', 'pdo', 'qbo', 'iod', 'solar']
comp_labelz = ['aao', 'amo', 'ao', 'enso', 'nao', 'pdo', 'qbo', 'iod', 'solar']

combined = []
for i, file in enumerate(filenames):
    df = open_sesame(file).to_dataframe(name=comp_labelz[i]).reset_index()
    df = df.drop(columns=['time'])
    combined.append(df)
Components = pd.concat(combined, axis=1)
Components['saod'] = xr.open_dataset(your_dir + '/data/Fitting Components/global_saod.nc')["Glossac_Aerosol_Optical_Depth"].to_dataframe(name='saod').reset_index()['saod']
Components['saod'] = Components['saod'] - Components['saod'].mean()
Components['pna'] = open_sesame('pna.data').to_dataframe(name='pna').reset_index().drop(columns=['time'])
Components = Components[0:264]
Components['trend'] = np.linspace(-131, 132, 264)
Components['sin_ann'] = np.sin(2*np.pi*np.linspace(-131, 132, 264)/12)
Components['cos_ann'] = np.cos(2*np.pi*np.linspace(-131, 132, 264)/12)
Components = Components[0:252]
trend_std = Components['trend'].std()
Components = (Components - Components.mean())/(Components.std())*Components['trend'].std()/120

### Multiple Linear Regression Analysis

#### Ordinary Least Squares

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
import statsmodels.api as sm
import pandas as pd

des_glob_model =  np.empty((4,10), dtype='object')
des_glob_modeltot = np.empty((4,10), dtype='object')
des_glob_residz = np.empty((4,10), dtype='object')
des_glob_outliers = np.empty((4,10), dtype='object')
des_glob_metric = np.empty((4,10), dtype='object')

global Components

# Ensure index is properly set on the original full dataframe
# If it's already a datetime/period index, it leaves it alone.
if not isinstance(Components.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    Components.index = pd.period_range(start='2002-09', periods=len(Components), freq='M')

for j in range(0,1):
    for i in range(9):
        if des_glob_meanz[j][i] is None:
            des_glob_model[j][i] = None
            continue

        x = des_glob_meanz[j][i]
        loc_mean = (x.shift(time=1) + x.shift(time=-1))/2
        outliers = np.abs(x - loc_mean) > 0.4
        des_glob_outliers[j][i] = outliers
        mask = outliers.values

        # Extract exact dates from the xarray dataset
        time_index = pd.to_datetime(x.time.values)
        
        # Perform alignment strictly for TRAINING the model
        comp_copy = Components.copy()
        if isinstance(comp_copy.index, pd.DatetimeIndex):
            comp_copy.index = comp_copy.index.to_period('M')
        comp_copy = comp_copy.groupby(level=0).mean()
        
        weighted = comp_copy.reindex(time_index.to_period('M'))
        weighted.index = time_index # Restore the exact dates
        weighted.loc[mask] = np.nan
        y_clean = x.where(~outliers)

        # Ensure we have overlapping data by checking NaNs in both y and X
        valid = ~np.isnan(y_clean.values) & ~weighted.isna().any(axis=1).values
        
        # If there's no overlapping data or too few samples, skip to avoid OLS errors
        if valid.sum() < weighted.shape[1] + 2:
            des_glob_model[j][i] = None
            continue

        X_clean = weighted.iloc[valid]
        y_vals = y_clean.values[valid]

        y_clean_series = pd.Series(y_vals, index=X_clean.index)

        des_glob_model[j][i] = sm.OLS(y_clean_series, X_clean).fit(cov_type='HAC', cov_kwds={'maxlags':1})

        # Reconstruct fitted values over the ENTIRE component dataset rather than cropping it
        weighted_full = Components.copy()
        for col in Components.columns:
            weighted_full[col] *= des_glob_model[j][i].params[col]
        
        modeltot = weighted_full.sum(axis=1)

        # Store the full-length model prediction
        des_glob_modeltot[j][i] = modeltot
        
        modeltot_aligned = modeltot.copy()
        if isinstance(modeltot_aligned.index, pd.DatetimeIndex):
            modeltot_aligned.index = modeltot_aligned.index.to_period('M')
        
        modeltot_aligned = modeltot_aligned.groupby(level=0).mean()
        modeltot_aligned = modeltot_aligned.reindex(time_index.to_period('M')).values
        
        resid_xr = x.copy()
        resid_xr.values = x.values - modeltot_aligned
        des_glob_residz[j][i] = resid_xr

        # ---- Manual CV RSE (OLS, excluding outliers) ----
        kfold = KFold(n_splits=5, shuffle=False)

        rse_in_list = []
        rse_out_list = []

        for train_idx, test_idx in kfold.split(X_clean):
            X_train, X_test = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
            y_train, y_test = y_vals[train_idx], y_vals[test_idx]
            tmp_model = sm.OLS(y_train, X_train).fit()
            ypred_train = tmp_model.predict(X_train)
            ypred_test = tmp_model.predict(X_test)
            rse_in_list.append(np.sqrt(np.mean((y_train - ypred_train)**2)))
            rse_out_list.append(np.sqrt(np.mean((y_test - ypred_test)**2)))

        RSE_in = np.mean(rse_in_list)
        RSE_out = np.mean(rse_out_list)

        r2 = 1 - np.nanvar(des_glob_residz[j][i]) / np.nanvar(x.values)
        n = np.sum(valid)
        p = X_clean.shape[1]
        r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

        des_glob_metric[j][i] = (RSE_in, RSE_out, r2, r2_adj)

print(f'Linreg for Variable {j} is complete!')

In [ ]:
#create a table of coefficients with confidence intervals for each component and each dataset, with components sorted by mean magnitude over all datasets

# Configuration
j = 0  # variable to analyze
sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']

# Collect coefficient rows
rows = {}
component_means = {}

for i in range(len(des_glob_model[j])):
    res = des_glob_model[j][i]
    if res is None:
        continue
    df = res.df_resid
    tcrit = ss.t.ppf(0.975, df)
    row = {}
    for comp in res.params.index:   # use actual columns from model
        beta = res.params[comp]
        se   = res.bse[comp]
        row[comp] = f"{beta:.3f} ± {tcrit*se:.3f}"
        component_means.setdefault(comp, []).append(beta)
    rows[sourcelabels[i]] = row

# Create DataFrame
table = pd.DataFrame.from_dict(rows, orient="index")

# Sort columns by mean magnitude
comp_order = sorted(component_means.keys(),key=lambda c: np.mean([abs(x) for x in component_means[c]]),reverse=True)
table = table[comp_order]

# Display / save
print(table)
table.to_csv("60N-60S_fit_table.csv")

In [ ]:
#labels to use for plotting
sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']
colors = ['red', 'tab:orange','y','forestgreen','dodgerblue','blue', 'darkorchid','magenta','tab:brown']
varlabels=['SAT Anomaly', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['Temperature [°K]', 'Specific Humidity [kg/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST','SH','850T','500T']

In [ ]:
#plot input, model, residual, and each component time series for each dataset; legend contains standard error estimates
plt.close('all')

for j in range(0,1):
    fig, axs = plt.subplots(5,2, figsize=(18, 20), dpi=600)
    axs = axs.flatten() 
    for i, k in enumerate([0,1,2,3,4,5,6,7,8]):
        x = des_glob_meanz[j][k]
        y = des_glob_model[j][k]
        if x is None or y is None:
            continue
        x = x[0:]
        # Plot the time series with datetime x-axis
        ax = axs[i]
        outliers = des_glob_outliers[j][k]
        mask = outliers.values  # boolean array
        # Break line at outliers
        x_line = x.where(~outliers)
        ax.plot(x['time'], x_line,label="Input Data",lw=1.5,color='black',zorder=2)
        # Plot ONLY the outlier points
        ax.scatter(x['time'].values[mask],x.values[mask],color='black',s=18,zorder=5)
        weighted = Components.copy().iloc[:len(des_glob_meanz[j][k])]
        m = 0
        for col in Components.columns:
            weighted[col] *= y.params[col]  # element-wise multiplication
            # Compute 95% CI
            df = y.df_resid
            tcrit = ss.t.ppf(0.975, df)
            half_width = tcrit * y.bse[col]
            lower_bound = y.params[col] - half_width
            # opacity based on lower bound
            opacity = np.clip(lower_bound / np.linalg.norm(np.abs(y.params))*2, 0, 1)
            ax.plot(x['time'], weighted[col], label=col+f' ({y.params[col]:.3f} ± {half_width:.3f})', alpha=opacity, lw=0.9)

        modelidea = weighted.sum(axis=1)
        ax.plot(x['time'], modelidea, label='Model', lw=1.5, color='blue', zorder=3)
        residual = x - modelidea
        res_line = residual.where(~outliers)
        ax.plot(x['time'], res_line,label='Residual',lw=1.5,color='red',zorder=4)
        # residual outlier dots
        ax.scatter(x['time'].values[mask],residual.values[mask],color='red',s=18,zorder=6)
        comp_names = [c for c in Components.columns if c in y.params.index]
        lower_bounds = {col: y.params[col] - tcrit*y.bse[col] for col in comp_names}
        sorted_comps = sorted(comp_names, key=lambda c: lower_bounds[c], reverse=True)
        comp_labels = [col + f' ({y.params[col]:.3f} ± {tcrit*y.bse[col]:.3f})' for col in sorted_comps]
        desired_labels = ['Input Data', 'Model', 'Residual'] + comp_labels
        handles, labels = ax.get_legend_handles_labels()
        label_to_handle = dict(zip(labels, handles))
        ordered_handles = [label_to_handle[l] for l in desired_labels if l in label_to_handle]
        ordered_labels = [l for l in desired_labels if l in label_to_handle]
        ax.legend(handles=ordered_handles, labels=ordered_labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
        RSE_in, RSE_out, r2_cv, r2_adj_cv = des_glob_metric[j][k]
        ax.set_title(sourcelabels[k] + "\n" + r"$R^2_{adj} = $" + f"{r2_adj_cv:.3f}" + r", $RMSE_{training} = $" + f"{RSE_in:.3f}" + r", $RMSE_{validation} = $" + f"{RSE_out:.3f}" + r", $\frac{RMSE_{validation}}{RMSE_{training}} = $" + f"{(RSE_out/RSE_in):.3f}")
        ax.set_ylabel(unitlabels[j])
        ax.minorticks_on()
        ax.grid(True, which='minor', alpha=0.3)
        ax.grid(True, which='major')
    fig.suptitle("60°N-60°S Mean "+varlabels[j]+" OLS Multiple Linear Regression\n", fontsize=18, y=0.99)
    fig.tight_layout()
    fig.savefig("New_60N-60S_"+filelabels[j]+"_HAC_LinearMod.png", dpi=600)
plt.close('all')

In [ ]:
# comparisons between datasets over plots of input, model, and residual

for j in range(0,1):
    fig, axs = plt.subplots(3,1, figsize=(15, 9), dpi=400, sharex=True)
    lines = []
    labels = []
    x_true = des_glob_meanz[0][0]['time']
    for i in range(9):
        if i == 1 or i==2 or i == 8:
            alph= 0.6
        elif i ==5 or i==7:
            alph = 0.45
        else:
            alph= 0.5
        x = des_glob_meanz[j][i]
        modeltot = des_glob_modeltot[j][i]
        residual = des_glob_residz[j][i]
        outliers = des_glob_outliers[j][i]
        mask = outliers.values
        # INPUT (break line at outliers)
        x_line = x.where(~outliers)
        l0, = axs[0].plot(x['time'], x_line, color=colors[i], label=sourcelabels[i], alpha=alph)        
        axs[0].scatter(x['time'].values[mask], x.values[mask], color=colors[i], s=18, zorder=5, alpha=alph)
        axs[1].plot(x_true, Components['trend']*des_glob_model[j][i].params['trend'], color=colors[i], alpha=alph)
        axs[1].plot(x_true, modeltot, color=colors[i], alpha=alph)
        # RESIDUAL (break line at outliers)
        res_line = residual.where(~outliers)
        axs[2].plot(x['time'], res_line, color=colors[i], alpha=alph)
        axs[2].scatter(x['time'].values[mask], residual.values[mask], color=colors[i], s=18, zorder=5, alpha=alph)
        lines.append(l0)
        RSE_in, RSE_out, r2_cv, r2_adj_cv = des_glob_metric[j][i]
        labels.append(sourcelabels[i] + "\n" + r"$R^2_{adj} = $" + f"{des_glob_model[j][i].rsquared_adj:.3f}\n"+ r"$\frac{RMSE_{validation}}{RMSE_{training}} = $" + f"{(RSE_out/RSE_in):.3f}")
    axs[0].set_title("Input Data")
    axs[1].set_title("Model")
    axs[2].set_title("Residual")
    for ax in axs:
        ax.minorticks_on()
        ax.grid(True, which='minor', alpha=0.3)
        ax.grid(True, which='major')

    leg = fig.legend(lines, labels, loc='lower center', fontsize=9, handlelength=2, handleheight=6.5, markerscale=1.5, ncol=9, frameon=False, bbox_to_anchor=(0.5, 0.003))
    for txt in leg.get_texts():
        txt.set_va('baseline')
    fig.suptitle("60°N-60°S Mean " + varlabels[j] + " OLS Multiple Linear Regression", fontsize=16, y=0.965) #60°N-60°S
    fig.tight_layout(rect=[0, 0.09, 1, 0.98])
    fig.savefig("New_60N-60S_" + filelabels[j] + "_LinearComp.png", dpi=400)

plt.close('all')

#### OLS Residual Analysis (QQ / PACF )

In [ ]:
#QQ Plot and Autocorrelation

#labels to use for plotting
sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']
colors = ['red', 'darkorange','gold','forestgreen','dodgerblue','blue', 'darkorchid','magenta','tab:brown']
varlabels=['SAT Anomaly', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['Temperature [°K]', 'Specific Humidity [kg/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST','SH','850T','500T']
plt.close('all')

from statsmodels.graphics.tsaplots import plot_pacf

for j in range(0, 1):
    fig, axs = plt.subplots(5, 4, figsize=(14, 14), dpi=200)  
    axs = axs.reshape(5, 4)  # 2D for row/col indexing
    for k, i in enumerate([0,1,2,3,4,5,6,7,8]):
        x = des_glob_residz[j][i]
        if x is None or len(x) == 0:
            continue
        row = k // 2
        col = (k % 2) * 2
        
        # Q–Q plot (left column)
        ax_qq = axs[row, col]
        outliers = des_glob_outliers[j][i]
        mask = outliers.values
        x_vals = x.values
        # Get the sorting index that probplot uses (ascending)
        sorted_idx = np.argsort(x_vals)
        # Sort both residuals and mask the same way
        sorted_mask = mask[sorted_idx]
        (osm, osr), (slope, intercept, r) = ss.probplot(x_vals, dist="norm", plot=None)
        # Reference line
        ax_qq.plot(osm, slope*osm + intercept, zorder=10, color='red')
        # Non-outliers
        ax_qq.scatter(osm[~sorted_mask], osr[~sorted_mask], s=8)
        # Outliers in black
        ax_qq.scatter(osm[sorted_mask], osr[sorted_mask], color='black', s=12, zorder=5)
        ax_qq.set_title(f"{sourcelabels[i]} Gaussian Q-Q")
        ax_qq.minorticks_on()
        ax_qq.grid(True, which='major', linestyle='-', linewidth=0.8)
        ax_qq.grid(True, which='minor', linestyle=':', linewidth=0.5)
        
        # PACF plot (right column)
        ax_pacf = axs[row, col+1]
        plot_pacf(x, lags=30, ax=ax_pacf, method="ywm")
        ax_pacf.set_title(f"{sourcelabels[i]} PACF")
        ax_pacf.minorticks_on()
        ax_pacf.grid(True, which='major', linestyle='-', linewidth=0.8)
        ax_pacf.grid(True, which='minor', linestyle=':', linewidth=0.5)
        
    fig.suptitle("Global Mean Residual Analysis for " + varlabels[j] + " (GLSAR)", 
                 fontsize=15, y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.99])
    fig.savefig("NNew_Global_"+filelabels[j] + "_GLSAR_QQ_PACF.png", dpi=400)
    plt.close(fig)

#### Ridge Regression with Cross Validation

In [ ]:
# ridge regression with cross-validation computation
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.model_selection import KFold

# -------------------------------
# Prepare storage arrays
# -------------------------------
des_glob_model =  np.empty((4,9), dtype='object')
des_glob_residz = np.empty((4,9), dtype='object')
des_glob_metric = np.empty((4,9), dtype='object')
des_glob_outliers = np.empty((4,9), dtype='object')   # <-- NEW

# Design matrix
X = Components.copy()

# Ridge alphas (include zero to mimic OLS)
alphas = np.logspace(-2, 2, 2000)
alphas = np.concatenate(([0], alphas))

# Loop over variables and datasets
for j in range(0,1):
    for i in range(9):
        y = des_glob_meanz[j][i]  # xarray.DataArray
        if y is None:
            des_glob_model[j][i] = None
            des_glob_outliers[j][i] = None   # outlier storage
            continue

        # ---- Outlier detection ----
        loc_mean = (y.shift(time=1) + y.shift(time=-1)) / 2
        outliers = np.abs(y - loc_mean) > 0.4
        des_glob_outliers[j][i] = outliers   # store outliers
        mask = outliers.values

        # ---- Prepare design matrix and clean y (fitting) ----
        X_std = X.copy().iloc[:len(y)]
        X_std.loc[mask] = np.nan
        y_clean = y.where(~outliers)
        valid = ~np.isnan(y_clean.values)
        X_clean = X_std.iloc[valid]
        y_vals = y_clean.values[valid]

        # ---- KFold for RidgeCV ----
        kfold = KFold(n_splits=5, shuffle=False)
        model = RidgeCV(alphas=alphas, cv=kfold, scoring='neg_root_mean_squared_error', fit_intercept=False)
        model.fit(X_clean, y_vals)

        # ---- Full-length prediction (NO NaNs) ----
        X_full = X.copy().iloc[:len(y)]
        yhat_full = model.predict(X_full)

        # ---- Residuals ----
        resid_full = xr.DataArray(y.values - yhat_full, dims=['time'], coords={'time': y['time']})
        des_glob_residz[j][i] = resid_full

        # ---- R^2 and adjusted R^2 ----
        r2 = 1 - np.nanvar(resid_full.values) / np.nanvar(y.values)
        n = np.sum(valid)
        p = X_clean.shape[1]
        r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

        # ---- Manual CV RSE ----
        rse_in_list = []
        rse_out_list = []
        for train_idx, test_idx in kfold.split(X_clean):
            X_train, X_test = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
            y_train, y_test = y_vals[train_idx], y_vals[test_idx]
            tmp_model = Ridge(alpha=model.alpha_, fit_intercept=False)
            tmp_model.fit(X_train, y_train)
            ypred_train = tmp_model.predict(X_train)
            ypred_test = tmp_model.predict(X_test)
            rse_in_list.append(np.sqrt(np.mean((y_train - ypred_train)**2)))
            rse_out_list.append(np.sqrt(np.mean((y_test - ypred_test)**2)))
        rse_in_avg = np.mean(rse_in_list)
        rse_out_avg = np.mean(rse_out_list)

        # ---- Save results ----
        des_glob_model[j][i] = model
        des_glob_metric[j][i] = (rse_in_avg, rse_out_avg, r2, r2_adj)

    print(f'RidgeCV for Variable {j} complete!')

In [ ]:
# plot table of ridge hyperparameter/coefficients, with components sorted by mean magnitude over all datasets

# Configuration
j = 0  # variable index

# Make a DataFrame for the table
rows = {}
component_names = Components.columns

# Collect coefficient rows
for i in range(len(des_glob_model[j])):
    model = des_glob_model[j][i]
    if model is None:
        continue
    row = {comp: model.coef_[k] for k, comp in enumerate(component_names)}
    row["Lambda"] = model.alpha_  # add lambda row
    rows[sourcelabels[i]] = row

# Create DataFrame
table = pd.DataFrame.from_dict(rows, orient="index")

# reorder components by mean magnitude
comp_order = sorted(component_names, key=lambda c: np.mean([abs(table.loc[src, c]) for src in table.index]),reverse=True)
table = table[comp_order + ["Lambda"]]  # add Lambda at the end

table = table.round(3)
print(table)
table.to_csv("ridge_regression_table.csv")

In [ ]:
#plot input, model, residual, and each component time series for each dataset; legend contains standard error estimates
plt.close('all')
for j in range(0,1):
    fig, axs = plt.subplots(5,2, figsize=(18, 20), dpi=600)
    axs = axs.flatten() 

    for k, i in enumerate([0,8,1,2,3,4,6,5,7]):
        x = des_glob_meanz[j][i]
        model = des_glob_model[j][i]
        metric = des_glob_metric[j][i]  
        if x is None or model is None or metric is None:
            continue
        x = x[0:]  # make a copy
        outliers = des_glob_outliers[j][i]
        mask = outliers.values
        RSE_in, RSE_out, r2, r2_adj = metric

        ax = axs[k]
        # Plot original input data
        x_line = x.where(~outliers)
        ax.plot(x['time'], x_line, label="Input Data", lw=1.5, color='black', zorder=2)
        ax.scatter(x['time'].values[mask], x.values[mask], color='black', s=18, zorder=5)

        # Weighted contributions from each component
        weighted = X.copy().iloc[:len(des_glob_meanz[j][i])]  # use the standardized X used for RidgeCV
        for col in X.columns:
            weighted[col] *= model.coef_[list(X.columns).index(col)]  # element-wise contribution

        # Rank components by absolute coefficient
        comp_names = list(X.columns)
        comp_abs = {c: abs(model.coef_[list(X.columns).index(c)]) for c in comp_names}
        sorted_comps = sorted(comp_names, key=lambda c: comp_abs[c], reverse=True)

        # Plot each component with its coefficient in the label
        for col in sorted_comps:
            coef = model.coef_[list(X.columns).index(col)]
            ax.plot(x['time'], weighted[col], label=f"{col} (c = {coef:.3f})", lw=0.9)

        # Total model and residual
        modelidea = weighted.sum(axis=1)
        ax.plot(x['time'], modelidea, label='Model', lw=1.5, color='blue', zorder=3)
        residual = x - modelidea
        res_line = residual.where(~outliers)
        ax.plot(x['time'], res_line, label='Residual', lw=1.5, color='red', zorder=4)
        ax.scatter(x['time'].values[mask], residual.values[mask], color='red', s=18, zorder=6)

        # Legend ordering: input, model, residual, then components by rank
        handles, labels = ax.get_legend_handles_labels()

        # Keep order: Input, Model, Residual, then components
        ordered_handles = []
        ordered_labels = []

        for h, l in zip(handles, labels):
            if l.startswith("Input Data"):
                ordered_handles.insert(0, h)
                ordered_labels.insert(0, l)
            elif l == "Model":
                ordered_handles.append(h)
                ordered_labels.append(l)
            elif l == "Residual":
                ordered_handles.append(h)
                ordered_labels.append(l)

        # Append components in sorted order
        for col in sorted_comps:
            for h, l in zip(handles, labels):
                if l.startswith(f"{col} ("):
                    ordered_handles.append(h)
                    ordered_labels.append(l)

        ax.legend(ordered_handles,ordered_labels,loc='upper center',bbox_to_anchor=(0.5, -0.15),ncol=4,frameon=False)

        # Title with optimal lambda and in/out-of-sample RSE
        alpha_opt = model.alpha_
        rse_in, rse_out, r2, r2_adj = metric
        ax.set_title(sourcelabels[i] + "\n" + r"$R^2_{adj}$ = "+ f"{r2_adj:.3f}, " + r"$\lambda$ = " + f"{alpha_opt:.3f}" + r", $RMSE_{training} = $" + f"{RSE_in:.3f}" + r", $RMSE_{validation} = $" + f"{RSE_out:.3f}" + r", $\frac{RMSE_{validation}}{RMSE_{training}} = $" + f"{(RSE_out/RSE_in):.3f}")
        ax.set_ylabel(unitlabels[j])
        ax.grid(True)

    fig.suptitle(f"Global Mean {varlabels[j]} Ridge Multiple Linear Regression\n", fontsize=18, y=0.99)
    fig.tight_layout()
    fig.savefig(f"New_Global_{filelabels[j]}_RidgeMod.png", dpi=600)

plt.close('all')


In [ ]:
# comparisons between datasets over plots of input, model, and residual

for j in range(0,1):
    fig, axs = plt.subplots(3,1, figsize=(15, 9), dpi=500, sharex=True)
    lines = []
    labels = []

    for i in range(9):
        x = des_glob_meanz[j][i]
        model = des_glob_model[j][i]
        metric = des_glob_metric[j][i]
        outliers = des_glob_outliers[j][i]
        if x is None or model is None or metric is None:
            continue
        RSE_in, RSE_out, r2, r2_adj = metric

        mask = outliers.values
        X_std = X.copy().iloc[:len(x)]

        # Reconstruct ridge model total
        weighted = X_std.copy()
        for k, col in enumerate(X.columns):
            weighted[col] *= model.coef_[k]
        modeltot = weighted.sum(axis=1)
        residual = x.values - modeltot.values


        if i == 1 or i==2 or i == 8:
            alph= 0.6
        elif i ==5 or i==7:
            alph = 0.45
        else:
            alph= 0.5
        # INPUT (break at outliers)
        x_line = x.where(~outliers)
        l0, = axs[0].plot(x['time'], x_line, label=sourcelabels[i], color=colors[i],alpha=alph)
        axs[0].scatter(x['time'].values[mask], x.values[mask], color=colors[i], s=18, zorder=5, alpha=alph)

        # MODEL
        axs[1].plot(x['time'], modeltot, color=colors[i], alpha=alph)
        trend_coef = des_glob_model[j][i].coef_[Components.columns.get_loc('trend')]
        axs[1].plot(x['time'], Components['trend']*trend_coef, color=colors[i], alpha=alph)

        # RESIDUAL (break at outliers)
        res_line = np.where(mask, np.nan, residual)
        axs[2].plot(x['time'], res_line, color=colors[i], alpha=alph)
        axs[2].scatter(x['time'].values[mask], residual[mask], color=colors[i], s=18, zorder=5, alpha=alph)

        rse_in, rse_out, r2, r2_adj = metric
        lines.append(l0)
        labels.append(sourcelabels[i] + "\n" + r"$R^2_{adj} = $" + f"{r2_adj:.3f}\n"+  r"$\lambda$ = " + f"{model.alpha_:.2f}\n" + r"$\frac{RMSE_{validation}}{RMSE_{training}} = $" + f"{(rse_out/rse_in):.3f}")

    axs[0].set_title("Input Data")
    axs[1].set_title("Model")
    axs[2].set_title("Residual")

    for ax in axs:
        ax.minorticks_on()
        ax.grid(True, which='minor', alpha=0.3)
        ax.grid(True, which='major')

    leg = fig.legend(lines, labels, loc='lower center', fontsize=9, handlelength=2, handleheight=6.5, markerscale=1.5, ncol=9, frameon=False, bbox_to_anchor=(0.5, 0.003))
    for txt in leg.get_texts():
        txt.set_va('baseline')

    fig.suptitle("Global Mean " + varlabels[j] + " Ridge Multiple Linear Regression", fontsize=16, y=0.965)
    fig.tight_layout(rect=[0, 0.1, 1, 0.98])
    fig.savefig("New_Global_" + filelabels[j] + "_RidgeComp.png", dpi=500)

plt.close('all')

### Changepoint Analysis

In [ ]:
import ruptures as rpt
from sklearn.linear_model import LinearRegression
from math import log

#### Simple Application

In [ ]:
# wrapper function to apply kernel changepoint detection with a given penalty and output the changepoints
def KernelCPDanal(airs_temp_mean, kernel, min_size, criterion='bic', gamma=None, printer=True):
    n = len(airs_temp_mean)
    if hasattr(airs_temp_mean, "coords"):  # xarray case
        x = airs_temp_mean.coords["time"].values
        y = airs_temp_mean.values
    else:  # numpy case
        n = len(airs_temp_mean)
        x = des_glob_meanz[0][0].coords["time"].values
        y = airs_temp_mean

    # Fit model
    if gamma == None:
        algo = rpt.KernelCPD(kernel=kernel, min_size=min_size).fit(y.reshape(-1,1))
    else:
        algo = rpt.KernelCPD(kernel=kernel, min_size=min_size, params={'gamma':gamma}).fit(y.reshape(-1,1))

    # Use BIC-style penalty directly
    if criterion == "aic":
        penalty = 1
    elif criterion == "bic":
        penalty = 1/2*np.log(n)
    elif criterion == "moderate":
        penalty = 2
    elif isinstance(criterion, (int, float)):
        penalty = criterion
    else:
        raise ValueError("criterion must be 'aic', 'bic', or 'moderate'")
    bkps = algo.predict(pen=penalty)
    if printer:
        print(f" BP: {len(bkps) - 1}, {[pd.to_datetime(x[xpos]).strftime('%Y-%m') for xpos in bkps[:-1]]}")

    return bkps

In [ ]:
# simple application and output

BIC_des_Kernel_Linear_BP =  np.empty((4,9), dtype='object')
#labels to use for plotting
sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']
colors = ['red', 'tab:orange','y','forestgreen','dodgerblue','blue', 'darkorchid','magenta','tab:brown']
varlabels=['SAT Anomaly', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['Temperature [°K]', 'Specific Humidity [kg/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST','SH','850T','500T']
varlabels=['Surface Air Temperature', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']

print("Global Mean Residual - KERNELCPD RBF (BIC Penalty)")
for j in range(0,1):
    print('\n' + varlabels[j])
    for i in range(9):
        if des_glob_meanz[j][i] is None:
            BIC_des_Kernel_Linear_BP[j][i] = None, None, None
            continue
        print(sourcelabels[i] + " ", end="")
        BIC_des_Kernel_Linear_BP[j][i] = KernelCPDanal(des_glob_meanz[j][i], "linear", 1, criterion='bic',printer=True) #no_clim_meanz des_glob_meanz des_glob_residz


In [ ]:
#plot changepoints in mean
sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']
colors = ['red', 'tab:orange','y','forestgreen','dodgerblue','blue', 'darkorchid','magenta','tab:brown']
varlabels=['SAT Anomaly', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['Temperature [°K]', 'Specific Humidity [kg/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST','SH','850T','500T']
for j in range(0,1):
    fig, axs = plt.subplots(5,2, figsize=(15, 11), dpi=300)
    axs = axs.flatten()
    axs[-1].axis('off')  # Turn off the last subplot if not used
    for k, i in enumerate([0,1,2,3,4,5,6,7,8]):
        if des_glob_meanz[j][i] is None:
            continue
        x, y, opt_bkps = des_glob_meanz[j][i].coords["time"].values, des_glob_meanz[j][i].values, BIC_des_Kernel_Linear_BP[j][i]
    # Plot the time series with datetime x-axis
        ax = axs[k]
        ax.plot(x, y, label="Time Series")
        ax.set_title(sourcelabels[i])
        ax.set_ylabel(unitlabels[j])
        ax.grid(True)
        # Draw vertical lines at the breakpoint datetimes
        for k in range(len(opt_bkps)):
            bkp = opt_bkps[k]
            bkprev = 0 if k == 0 else opt_bkps[k-1]
            # Segment mean line using ax.plot
            seg_x = x[bkprev:bkp]
            seg_y = [np.mean(y[bkprev:bkp])] * len(seg_x)
            ax.plot(seg_x, seg_y, color="orange", linestyle="-", linewidth=2, label="Segment Mean" if k == 0 else "")
            if k != len(opt_bkps)-1 and len(opt_bkps) != 1:
                xpos = x[bkp]
                ax.axvline(x=xpos, color="red", linestyle="--", label="Breakpoint" if i == 0 else "")
                ax.text(x[bkp+1], ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) / 20, pd.to_datetime(xpos).strftime("%Y-%m"), rotation=90, fontsize=8, color='red', ha='left')
    fig.suptitle("Global Mean SAT Anomaly Changepoints in Mean \n (Full Time Series, Linear Kernel, BIC Penalty)", fontsize=15)
    fig.tight_layout()
    fig.savefig("Global"+filelabels[j]+"_BIC_BP_mean.png", dpi=300)

#### Main Sliding Window Functions

In [ ]:
import ruptures as rpt
from sklearn.linear_model import LinearRegression
from math import log

In [ ]:
# wrapper function to apply kernel changepoint detection with a given penalty and output the changepoints
def KernelCPDanal(airs_temp_mean, kernel, min_size, criterion='bic', gamma=None, printer=True):
    n = len(airs_temp_mean)
    if hasattr(airs_temp_mean, "coords"):  # xarray case
        x = airs_temp_mean.coords["time"].values
        y = airs_temp_mean.values
    else:  # numpy case
        n = len(airs_temp_mean)
        x = des_glob_meanz[0][0].coords["time"].values
        y = airs_temp_mean

    # Fit model
    if gamma == None:
        algo = rpt.KernelCPD(kernel=kernel, min_size=min_size).fit(y.reshape(-1,1))
    else:
        algo = rpt.KernelCPD(kernel=kernel, min_size=min_size, params={'gamma':gamma}).fit(y.reshape(-1,1))

    # Use BIC-style penalty directly
    if criterion == "aic":
        penalty = 1
    elif criterion == "bic":
        penalty = 1/2*np.log(n)
    elif criterion == "moderate":
        penalty = 2
    elif isinstance(criterion, (int, float)):
        penalty = criterion
    else:
        raise ValueError("criterion must be 'aic', 'bic', or 'moderate'")
    bkps = algo.predict(pen=penalty)
    if printer:
        print(f" BP: {len(bkps) - 1}, {[pd.to_datetime(x[xpos]).strftime('%Y-%m') for xpos in bkps[:-1]]}")

    return bkps

In [ ]:
def slidingwindow(dataset, window, func, step=1, align='center'):
    """
    Sliding window over xarray DataArray along 'time', accepts integer outputs.
    Returns total and average per time index.

    Parameters
    ----------
    dataset : xarray.DataArray
        DataArray with a 'time' dimension.
    window : int
        Window length (number of indices)
    func : callable
        Function applied to each window. Must return integer array of length `window`.
    step : int
        Step size for sliding the window.
    align : str
        'center' or 'start' alignment of output (unused in this version but kept for consistency).

    Returns
    -------
    total : xarray.DataArray
        Total sum of func outputs at each time index (before averaging)
    avg : xarray.DataArray
        Average per time index (total / number of windows covering that index)
    """
    n = dataset.sizes['time']
    total = np.zeros(n, dtype=float)
    count_windows = np.zeros(n, dtype=float)

    for start in range(0, n - window + 1, step):
        end = start + window
        win_data = dataset.isel(time=slice(start, end))
        result = func(win_data)
        if isinstance(result, xr.DataArray):
            result = result.values
        result = np.array(result, dtype=float)
        if len(result) != window:
            raise ValueError(f"Function output length {len(result)} does not match window length {window}")

        total[start:end] += result
        count_windows[start:end] += 1

    avg = total / np.maximum(count_windows, 1)
    return xr.DataArray(total, coords={'time': dataset.time}, dims='time'), \
           xr.DataArray(avg, coords={'time': dataset.time}, dims='time')


In [ ]:
def KernelCPD_int(win_data, kernel="rbf", min_size=1, criterion="bic"):
    """
    Returns integer array: 1 if breakpoint at index, 0 otherwise
    """
    bkps = KernelCPDanal(win_data, kernel=kernel, min_size=min_size, criterion=criterion, printer=False)
    n = len(win_data)
    arr = np.zeros(n, dtype=int)
    for b in bkps[:-1]:  # exclude the last index
        if b < n:
            arr[b] = 1
    return arr

def KernelCPDmaxbreaks(airs_temp_mean, x):
    if airs_temp_mean is None:
        return None
    n = len(airs_temp_mean)
    # Run KernelCPDanal for all penalties
    y = [KernelCPDanal(airs_temp_mean, "rbf", 1, criterion=crit, gamma=None, printer=False) for crit in x]

    # Initialize max_penalty array for all time indices
    max_penalty = np.zeros(n)

    # For each time index, find max penalty where it occurs
    for t in range(n):
        # Boolean array: True if t is in the breakpoints for each penalty
        exists = np.array([t in res for res in y])
        if np.any(exists):
            max_penalty[t] = x[np.where(exists)[0].max()]
        else:
            max_penalty[t] = 0  # never occurs
    return y, max_penalty

def KernelCPDmax_int(win_data, x=np.linspace(0.1, 20, 200)):
    """
    Returns max_penalty array (integers/floats)
    """
    _, max_penalty = KernelCPDmaxbreaks(win_data, x)
    return max_penalty  # already numeric per time index

In [ ]:
# apply sliding window using parallelization
from concurrent.futures import ProcessPoolExecutor

windowmaxpen = np.empty((4, 9), dtype='object')
windowbpks   = np.empty((4, 9), dtype='object')

def compute_for_dataset(args):
    j, i, residz = args
    if residz is None:
        return j, i, (None, None), (None, None)
    bpks = slidingwindow(residz, 120, KernelCPD_int, step=1, align='center')
    maxpen = slidingwindow(residz, 120, KernelCPDmax_int, step=1, align='center')
    print(f"Computed for var{j}, dataset {i}")
    return j, i, bpks, maxpen

# --- Collect work tasks ---
tasks = []
for j in range(0,1):
    for i in range(9):
        tasks.append((j, i, des_glob_residz[j][i])) #this is where you can change the input 

with ProcessPoolExecutor(max_workers=80) as exe:
    for j, i, bpks, maxpen in exe.map(compute_for_dataset, tasks):
        windowbpks[j][i] = bpks
        windowmaxpen[j][i] = maxpen

#### Visualization and Saving

In [ ]:
sourcelabels = ['AIRSv7', 'CLIMCAPS-AQUA','MERRA2',  'ERA5', 'JRA55', 'JRA3Q', 'BEST', 'GISTEMP', 'AIRSv7 Skin']
colors = ['red', 'tab:orange','y','forestgreen','dodgerblue','blue', 'darkorchid','magenta','tab:brown']
varlabels=['SAT Anomaly', 'Surface Specific Humidity', '850 hPa Temperature', '500 hPa Temperature']
unitlabels=['Temperature [°K]', 'Specific Humidity [kg/kg]', 'Temperature [°K]','Temperature [°K]']
filelabels=['ST','SH','850T','500T']

In [ ]:
# save observed changepoint score and frequency as netcdf for future analysis
def tuple_to_dataset(da_tuple, names=("a", "b")):
    return xr.Dataset({names[0]: da_tuple[0],names[1]: da_tuple[1],})

for j in range(windowbpks.shape[0]):
    for i in range(windowbpks.shape[1]):
        if windowbpks[j, i] is None:
            continue

        ds_bpks = tuple_to_dataset(windowbpks[j, i], names=("bpks_count", "bpks_score"))
        ds_maxp = tuple_to_dataset(windowmaxpen[j, i], names=("maxpen_count", "maxpen_score"))

        ds_bpks.to_netcdf(fyour_dir + "/data/breakpointsaved/60N-60S/windowbpks_var{j}_ds{i}.nc")
        ds_maxp.to_netcdf(fyour_dir + "/data/breakpointsaved/60N-60S/windowmaxpen_var{j}_ds{i}.nc")

In [ ]:
N = 252
dates = pd.date_range("2002-09", periods=N, freq="MS")

def align_to_master(arr, x_dates, master_dates):
    """
    Align array to the master dates by mapping the specific time index.
    This safely anchors shorter/differing length time series to their exact real-world months.
    """
    if arr is None:
        return np.full(len(master_dates), np.nan)
    arr = np.array(arr)
    x_periods = pd.to_datetime(x_dates).to_period('M')
    master_periods = master_dates.to_period('M')
    
    # If the array length perfectly matches the time index, use pandas to align perfectly
    if len(arr) == len(x_periods):
        s = pd.Series(arr, index=x_periods)
        s = s.groupby(level=0).mean() # Safety net for duplicate indices
        return s.reindex(master_periods).values
    else:
        # Fallback if the array is slightly shorter (e.g., lost edges from sliding window)
        # We calculate the exact integer offset from the master start date (2002-09)
        offset = (x_periods[0].year - master_periods[0].year) * 12 + (x_periods[0].month - master_periods[0].month)
        pad_left = max(0, offset)
        pad_right = len(master_dates) - len(arr) - pad_left
        
        # Safety catch if array is somehow too long
        if pad_right < 0:
            arr = arr[:len(master_dates) - pad_left]
            pad_right = 0
            
        return np.pad(arr, (pad_left, pad_right), constant_values=np.nan)

# --- Plotting loop ---
for j in range(0, 1):  # or range(4) for all variables
    fig, axs = plt.subplots(2, 1, figsize=(6.5, 6), dpi=600)
    #fig, axs = plt.subplots(5, 2, figsize=(10, 12), dpi=600)
    axs = axs.flatten()
    #axs[-1].remove()

    for k, i in enumerate([1,2]):

    #for k, i in enumerate([0,1,2,3,4,5,6,7,8]):
        if windowbpks[j][i] is None or windowmaxpen[j][i] is None:
            continue

        # Get original x to extract its exact dates for alignment
        x = des_glob_meanz[j][i]
        x_dates = x.time.values
        # Extract and precisely map all four statistics to the master N=252 timeline
        bkp_total = align_to_master(windowbpks[j][i][0], x_dates, dates)
        bkp_avg   = align_to_master(windowbpks[j][i][1], x_dates, dates)
        pen_total = align_to_master(windowmaxpen[j][i][0], x_dates, dates)
        pen_avg   = align_to_master(windowmaxpen[j][i][1], x_dates, dates)
        ax = axs[k]
        ax.vlines(np.arange(N), 0, bkp_avg,   color="blue",  alpha=0.9, label="BIC Frequency", zorder=4)
        ax.vlines(np.arange(N), 0, pen_avg,   color="orange", alpha=0.9, label="Average Max Penalty", zorder=3)

        # ---- Label peaks where frequency > 0.2 but collapse nearby indices ----

        # np.where safely ignores NaNs (evaluates as False)
        valid_idx = np.where(bkp_avg >= 0.2)[0]  
        clusters = []
        current_cluster = []

        # --- Group indices within 2 time steps ---
        for t in valid_idx:
            if not current_cluster:
                current_cluster = [t]
            else:
                if t - current_cluster[-1] <= 2:
                    current_cluster.append(t)
                else:
                    clusters.append(current_cluster)
                    current_cluster = [t]

        # don't forget the last one
        if current_cluster:
            clusters.append(current_cluster)
        # --- From each cluster choose the index with the highest peak penalty ---
        peak_indices = []
        for cluster in clusters:
            # choose t that maximizes penalty height
            best_t = max(cluster, key=lambda t: pen_avg[t])
            peak_indices.append(best_t)
        # --- Now plot only selected peaks ---
        for t_idx in peak_indices:
            ax.text(t_idx,min(pen_avg[t_idx],7),dates[t_idx].strftime("%Y-%m"),color="blue",fontsize=5,ha="right",va="top",rotation=90)


        # Axis labels and date ticks
        # -----------------------------------------------------------
        ax.set_ylabel("Frequency / Penalty")
        yearly = np.arange(0, N, 24)
        ax.set_xticks(yearly)
        label_locs = np.arange(0, N, 48)
        label_vals = dates[::48].strftime("%Y-%m")

        labels = []
        label_i = 0
        for k in range(len(yearly)):
            if yearly[k] in label_locs:
                labels.append(label_vals[label_i])
                label_i += 1
            else:
                labels.append("")
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_xticks(np.arange(0, N, 3), minor=True)         # Minor ticks every 3 months
        ax.set_ylim(0, 7)  # adjust if needed
        ax.axhline(y=np.log(N-1)/2, color='green', linestyle='-', alpha=0.8, label='BIC Penalty',zorder=2)         # BIC reference lines
        ax.axhline(y=1, color='blue', linestyle='--', alpha=0.8, label='Maximum BIC Frequency',zorder=2)         # BIC reference lines
        ax.set_title(sourcelabels[i])
        ax.grid(True)
    fig.suptitle(f"60°N-60°S Mean {varlabels[j]}: Kernel Changepoint Detection \n (OLS Residual, RBF Kernel, 10-year Sliding Window)",fontsize=13.5,y=0.99)
    #fig.suptitle(f"Global Mean {varlabels[j]}: Kernel Changepoint Detection \n (OLS Residual, Linear Kernel, Full Time Series)",fontsize=16,y=0.995)
    #fig.suptitle(f"Global Mean {varlabels[j]}: Kernel Changepoint Detection \n (Input Data, RBF Kernel, 10-year Sliding Window)",fontsize=16,y=0.995)
    handles, labels = axs[0].get_legend_handles_labels()
    #fig.legend(handles,labels,loc="lower center",ncol=2,fontsize=12,bbox_to_anchor=(0.5, 0.00))
    fig.legend(handles,labels,loc="lower center",ncol=2,fontsize=9,bbox_to_anchor=(0.5, 0.000))
    fig.tight_layout(rect=[0,0.1,1,1])
    fig.savefig(f"T_60N-60S_10-year_KernelCPD_SlidingWindow_OLSResidual_RBF_var{varlabels[j]}.png",dpi=600)
    plt.show()